<a href="https://colab.research.google.com/github/SIRLLON/TP2-pb/blob/main/tp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP 2 (TP2)
**Aluno:** Sirllon
**Plataforma:** Google Colab Notebook  
Resolução comentada de concorrência, ordenação, estruturas lineares, recursividade e paralelismo com OpenMP.

### Exercício 1: Coleta de Dados com Asyncio
Executa 5 tarefas assíncronas em paralelo e agrega os resultados.

In [ ]:
import asyncio
import random

async def coletar_dados(id_tarefa):
    # Simula espera de rede
    await asyncio.sleep(random.uniform(0.1, 0.5))
    return f"Dados {id_tarefa}"

async def main_ex1():
    # Cria lista de tarefas
    tarefas = [coletar_dados(i) for i in range(1, 6)]
    # Aguarda todas as tarefas
    resultados = await asyncio.gather(*tarefas)
    print("Resultados agregados:", resultados)

await main_ex1()

### Exercício 2: asyncio.to_thread()
Mede a diferença entre chamar uma função bloqueante diretamente ou encapsulada em uma thread do event loop.

In [ ]:
import asyncio
import time

def funcao_custosa():
    # Simula carga CPU sincrona
    time.sleep(1)
    return "Finalizado"

async def main_ex2():
    # Mede tempo direto
    t_ini = time.time()
    funcao_custosa()
    print(f"Direto bloqueante: {time.time() - t_ini:.3f} s")

    # Mede com to_thread
    t_ini = time.time()
    await asyncio.to_thread(funcao_custosa)
    print(f"Com to_thread: {time.time() - t_ini:.3f} s")

await main_ex2()

### Exercício 3: Pipeline Assíncrono (E1 -> E2 -> E3)
Implementação de pipeline em fluxo que processa itens concorrentemente.

In [ ]:
import asyncio
import time

async def coleta(item_id):
    # Coleta dados assincronamente
    await asyncio.sleep(0.1)
    return f"dados_{item_id}"

def transformacao(dados):
    # Converte para maiusculas
    return dados.upper()

async def envio(dados):
    # Envia dados assincronamente
    await asyncio.sleep(0.1)
    print(f"Enviado: {dados}")

async def processar_item(item_id):
    # Fluxo do pipeline
    item = await coleta(item_id)
    trans = transformacao(item)
    await envio(trans)

async def main_ex3():
    # Processa multiplos itens
    await asyncio.gather(*(processar_item(i) for i in range(1, 4)))

await main_ex3()

### Exercício 4: Deadlock asyncio vs OpenMP (Explicação)
Um **deadlock** ocorre quando uma tarefa do `asyncio` bloqueia a thread única do *event loop* esperando que threads secundárias do `OpenMP` concluam uma região paralela. Ao mesmo tempo, se alguma thread do `OpenMP` precisar submeter um callback ou interagir com o event loop (aguardando resposta), ela ficará travada eternamente, pois a thread do event loop está bloqueada aguardando o fim do OpenMP.

### Exercício 5: Servidor TCP Eco com asyncio
Servidor de eco que recebe dados e retorna a mesma string prefixada.

In [ ]:
import asyncio

async def lidar_cliente(reader, writer):
    # Le dados enviados
    dados = await reader.read(100)
    mensagem = dados.decode()
    # Retorna o eco
    writer.write(f"ECO: {mensagem}".encode())
    await writer.drain()
    writer.close()

async def main_ex5():
    # Inicia o servidor
    servidor = await asyncio.start_server(lidar_cliente, '127.0.0.1', 8889)
    print("Servidor TCP rodando em 127.0.0.1:8889")
    async with servidor:
        await asyncio.sleep(0.2) # Mantem ativo por instantes
        servidor.close()

await main_ex5()

### Exercício 6: QuickSelect
Encontra o k-ésimo menor elemento de um vetor.

In [ ]:
def partition(arr, l, r):
    # Define pivo simples
    pivot = arr[r]
    i = l
    for j in range(l, r):
        if arr[j] <= pivot:
            arr[i], arr[j] = arr[j], arr[i]
            i += 1
    arr[i], arr[r] = arr[r], arr[i]
    return i

def quickselect(arr, l, r, k):
    # Algoritmo de selecao
    if l <= r:
        p_idx = partition(arr, l, r)
        if p_idx == k:
            return arr[k]
        elif p_idx > k:
            return quickselect(arr, l, p_idx - 1, k)
        else:
            return quickselect(arr, p_idx + 1, r, k)
    return -1

vetor = [12, 3, 5, 7, 4, 19, 26]
print("3º menor:", quickselect(vetor.copy(), 0, len(vetor)-1, 2))

### Exercício 7: QuickSelect com Histórico de Pivôs
Ajusta a seleção do pivô com base em histórico.

In [ ]:
# Memoriza ultimos pivos
historico_pivos = []

def escolhe_pivo_historico(arr, l, r, k_hist):
    # Escolhe pivo considerando historico
    p_idx = r
    if len(historico_pivos) >= k_hist:
        # Usa a mediana do historico
        p_idx = int(sum(historico_pivos[-k_hist:]) / k_hist) % (r - l + 1) + l
    historico_pivos.append(p_idx)
    return p_idx

### Exercício 8: QuickSort Mediana de Três vs Simples
Compara QuickSort simples com pivotagem de mediana.

In [ ]:
def quicksort_simples(arr):
    # QuickSort basico recursivo
    if len(arr) <= 1: return arr
    piv = arr[len(arr)//2]
    esq = [x for x in arr if x < piv]
    meio = [x for x in arr if x == piv]
    dir = [x for x in arr if x > piv]
    return quicksort_simples(esq) + meio + quicksort_simples(dir)

def mediana_de_tres(arr, l, r):
    mid = (l + r) // 2
    vals = [(arr[l], l), (arr[mid], mid), (arr[r], r)]
    vals.sort()
    return vals[1][1]

def quicksort_m3(arr, l, r):
    # QuickSort mediana de 3
    if l < r:
        p_idx = mediana_de_tres(arr, l, r)
        arr[p_idx], arr[r] = arr[r], arr[p_idx]
        p = partition(arr, l, r)
        quicksort_m3(arr, l, p - 1)
        quicksort_m3(arr, p + 1, r)

### Exercício 9: QuickSort com Dois Pivôs (Explicação)
O **Dual-Pivot QuickSort** divide o vetor em três partes usando dois pivôs $P_1$ e $P_2$ (onde $P_1 \le P_2$). Ele classifica os elementos em: menores que $P_1$, entre $P_1$ e $P_2$, e maiores que $P_2$.  
* **Vantagens**: Reduz o número de acessos e trocas de memória na prática, otimizando o cache.  
* **Desvantagens**: Particionamento e controle de índices significativamente mais complexos.

### Exercício 10: QuickSort Híbrido com Insertion Sort
Evita recursão profunda para pequenos subvetores (< 10 elementos).

In [ ]:
def insertion_sort(arr, l, r):
    # Ordenacao por insercao
    for i in range(l + 1, r + 1):
        key = arr[i]
        j = i - 1
        while j >= l and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key

def quicksort_hibrido(arr, l, r):
    # QuickSort com Insertion Sort
    if l < r:
        if r - l < 10:
            insertion_sort(arr, l, r)
        else:
            p = partition(arr, l, r)
            quicksort_hibrido(arr, l, p - 1)
            quicksort_hibrido(arr, p + 1, r)

### Exercício 11: Operações Básicas de Lista Encadeada
Implementação de nó e lista encadeada simples.

In [ ]:
class Node:
    def __init__(self, val):
        self.val = val
        self.next = None

class ListaEncadeada:
    def __init__(self):
        self.head = None

    def inserir_inicio(self, val):
        # Adiciona no inicio
        novo = Node(val)
        novo.next = self.head
        self.head = novo

    def inserir_fim(self, val):
        # Adiciona no fim
        novo = Node(val)
        if not self.head:
            self.head = novo
            return
        curr = self.head
        while curr.next: curr = curr.next
        curr.next = novo

    def remover(self, val):
        # Remove um elemento
        curr = self.head
        prev = None
        while curr:
            if curr.val == val:
                if prev: prev.next = curr.next
                else: self.head = curr.next
                return True
            prev = curr
            curr = curr.next
        return False

    def buscar(self, val):
        # Busca por valor
        curr = self.head
        while curr:
            if curr.val == val: return True
            curr = curr.next
        return False

### Exercício 12: Lista com Blocos de B Elementos (Explicação)
Uma **lista de blocos contíguos de tamanho B** (Unrolled Linked List) armazena múltiplos elementos contíguos em cada nó.
* **Vantagens**: Excelente localidade de cache e redução do número de ponteiros necessários na memória.
* **Desvantagens**: Inserções/remoções internas requerem deslocamento de dados complexos ou cisão (splitting) de nós.

### Exercício 13: Lista Duplamente Encadeada Ordenada
Mantém elementos em ordem com ponteiros bidirecionais.

In [ ]:
class DoublyNode:
    def __init__(self, val):
        self.val = val
        self.next = None
        self.prev = None

class DoublyLinkedListSorted:
    def __init__(self):
        self.head = None

    def inserir(self, val):
        # Insere mantendo a ordenacao
        novo = DoublyNode(val)
        if not self.head:
            self.head = novo
            return novo
        curr = self.head
        while curr and curr.val < val:
            if not curr.next:
                curr.next = novo
                novo.prev = curr
                return novo
            curr = curr.next
        if curr == self.head:
            novo.next = self.head
            self.head.prev = novo
            self.head = novo
        else:
            p = curr.prev
            p.next = novo
            novo.prev = p
            novo.next = curr
            curr.prev = novo
        return novo

    def remover_nodo(self, node):
        # Remove nodo por referencia
        if not node: return
        if node == self.head:
            self.head = node.next
        if node.next: node.next.prev = node.prev
        if node.prev: node.prev.next = node.next

### Exercício 14: Lista Circular
Estrutura encadeada onde o último nó aponta para o primeiro.
**Casos de uso reais**:
1. Escalonamento de Processos Round-Robin em Sistemas Operacionais.
2. Gerenciamento de turnos em jogos de tabuleiro multiplayer digitais.

In [ ]:
class CircularList:
    def __init__(self):
        self.head = None

    def inserir(self, val):
        # Insere na lista circular
        novo = Node(val)
        if not self.head:
            self.head = novo
            novo.next = novo
            return
        curr = self.head
        while curr.next != self.head:
            curr = curr.next
        curr.next = novo
        novo.next = self.head

### Exercício 15: Tabela Hash com Lista Encadeada para Colisão
Armazena pares chave-valor resolvendo colisões com encadeamento.

In [ ]:
class HashNode:
    def __init__(self, key, val):
        self.key = key
        self.val = val
        self.next = None

class HashTableChaining:
    def __init__(self, size=10):
        self.size = size
        self.buckets = [None] * size

    def _hash(self, key):
        return hash(key) % self.size

    def inserir(self, key, val):
        # Insere par chave valor
        idx = self._hash(key)
        novo = HashNode(key, val)
        if not self.buckets[idx]:
            self.buckets[idx] = novo
        else:
            curr = self.buckets[idx]
            while curr:
                if curr.key == key:
                    curr.val = val
                    return
                if not curr.next:
                    curr.next = novo
                    return
                curr = curr.next

    def buscar(self, key):
        # Busca por chave
        idx = self._hash(key)
        curr = self.buckets[idx]
        while curr:
            if curr.key == key: return curr.val
            curr = curr.next
        return None

### Exercício 16: Soma Recursiva de Vetor
Soma os n primeiros elementos do vetor.

In [ ]:
def soma_recursiva(arr, n):
    # Soma recursiva de vetor
    if n <= 0: return 0
    return arr[n-1] + soma_recursiva(arr, n-1)

print("Soma [1,2,3,4]:", soma_recursiva([1, 2, 3, 4], 4))

### Exercício 17: Busca Binária Recursiva
Encontra elemento no vetor ordenado recursivamente.

In [ ]:
def busca_binaria_rec(arr, x, l, r):
    # Busca binaria recursiva
    if l > r: return -1
    mid = (l + r) // 2
    if arr[mid] == x: return mid
    elif arr[mid] > x: return busca_binaria_rec(arr, x, l, mid-1)
    else: return busca_binaria_rec(arr, x, mid+1, r)

print("Busca 5 em [1,3,5,7]:", busca_binaria_rec([1, 3, 5, 7], 5, 0, 3))

### Exercício 18: Busca Binária Recursiva Híbrida (Iterativa para Profundidade > D)
Troca para modo iterativo se a pilha de recursão atingir um limite D.

In [ ]:
def busca_binaria_hibrida(arr, x, l, r, prof, D):
    # Troca de modo se prof >= D
    if prof >= D:
        # Roda busca iterativa
        while l <= r:
            mid = (l + r) // 2
            if arr[mid] == x: return mid
            elif arr[mid] > x: r = mid - 1
            else: l = mid + 1
        return -1
    else:
        if l > r: return -1
        mid = (l + r) // 2
        if arr[mid] == x: return mid
        elif arr[mid] > x: return busca_binaria_hibrida(arr, x, l, mid-1, prof+1, D)
        else: return busca_binaria_hibrida(arr, x, mid+1, r, prof+1, D)

### Exercício 19: Torre de Hanoi
Resolve recursivamente a transferência de discos.

In [ ]:
def hanoi(n, orig, dest, aux):
    # Resolve torre de hanoi
    if n == 1:
        print(f"Mover disco 1 de {orig} para {dest}")
        return
    hanoi(n-1, orig, aux, dest)
    print(f"Mover disco {n} de {orig} para {dest}")
    hanoi(n-1, aux, dest, orig)

hanoi(3, 'A', 'C', 'B')

### Exercício 20: Contagem de Dígitos Recursiva
Calcula quantos dígitos possui um número positivo.

In [ ]:
def contar_digitos(n):
    # Conta digitos recursivamente
    if n < 10: return 1
    return 1 + contar_digitos(n // 10)

print("Digitos de 1459:", contar_digitos(1459))

## Programação Concorrente em Python (Exercícios 21 ao 25)

Nesta seção, implementamos os exercícios de paralelismo usando a linguagem Python com a biblioteca `ThreadPoolExecutor` do módulo padrão `concurrent.futures`, em substituição ao OpenMP em C++. Isso simplifica a execução e teste diretamente no ambiente de execução do Colab.

### Exercício 21: Soma de Elementos com Parallel For + Reduction
Implementar a soma dos elementos de um vetor dividindo a carga de trabalho entre threads. Os resultados locais de cada thread são reduzidos (somados) na thread principal.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def soma_paralela(vetor, num_threads=4):
    n = len(vetor)
    # Divide o trabalho igualmente em blocos para cada thread
    tamanho_bloco = (n + num_threads - 1) // num_threads

    def somar_bloco(inicio):
        fim = min(inicio + tamanho_bloco, n)
        soma_local = 0
        for i in range(inicio, fim):
            soma_local += vetor[i]
        return soma_local

    inicios = [i * tamanho_bloco for i in range(num_threads)]

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        # Mapeia a execução da função somar_bloco em paralelo
        resultados = list(executor.map(somar_bloco, inicios))

    # Redução: combina os resultados parciais
    return sum(resultados)

# Testar a implementação
vetor_teste = [1] * 1000000
t_ini = time.time()
resultado = soma_paralela(vetor_teste, num_threads=4)
print(f"Soma paralela (4 threads): {resultado} | Tempo: {time.time() - t_ini:.4f} s")


### Exercício 22: Distribuição de Blocos da Lista por Threads
Implementar uma abordagem em que cada thread processa um conjunto de blocos da lista (distribuição em formato round-robin de blocos de dados entre threads).

In [ ]:
def processar_blocos_round_robin(lista_de_blocos, num_threads=4):
    def processar_grupo_de_blocos(id_thread):
        soma_local = 0
        # Cada thread consome blocos com passo fixo (round-robin) para balanceamento
        for i in range(id_thread, len(lista_de_blocos), num_threads):
            bloco = lista_de_blocos[i]
            soma_local += sum(bloco)
        return soma_local

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        resultados = list(executor.map(processar_grupo_de_blocos, range(num_threads)))

    return sum(resultados)

# Testar a implementação com blocos de dados
blocos_teste = [[i] * 1000 for i in range(50)]
resultado = processar_blocos_round_robin(blocos_teste, num_threads=4)
print(f"Soma com distribuição de blocos (round-robin): {resultado}")


### Exercício 23: Balanceamento de Carga com Prefix Sums
Implementar política de balanceamento de carga baseada na soma de prefixos (prefix sums) para distribuir blocos de tamanhos variados uniformemente entre as threads.

In [ ]:
import bisect
import random

def prefix_sums_balance(lista_de_blocos, num_threads=4):
    tamanhos = [len(bloco) for bloco in lista_de_blocos]

    # Calcular soma de prefixos (prefix sums) do esforço de trabalho
    prefix_sums = [0]
    soma_acumulada = 0
    for tam in tamanhos:
        soma_acumulada += tam
        prefix_sums.append(soma_acumulada)

    total_trabalho = prefix_sums[-1]
    trabalho_por_thread = total_trabalho / num_threads

    # Encontrar as divisões dos blocos usando busca binária (bisect)
    divisoes = [0]
    for i in range(1, num_threads):
        alvo = i * trabalho_por_thread
        idx = bisect.bisect_left(prefix_sums, alvo)
        # Ajustar para o bloco mais próximo da divisão ideal de carga
        if idx > 1 and abs(prefix_sums[idx-1] - alvo) < abs(prefix_sums[idx] - alvo):
            idx = idx - 1
        divisoes.append(idx)
    divisoes.append(len(lista_de_blocos))

    # Função para cada thread processar seu intervalo de blocos balanceado
    def processar_intervalo(id_thread):
        inicio_bloco = divisoes[id_thread]
        fim_bloco = divisoes[id_thread + 1]
        soma_local = 0
        for b_idx in range(inicio_bloco, fim_bloco):
            soma_local += sum(lista_de_blocos[b_idx])
        return soma_local

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        resultados = list(executor.map(processar_intervalo, range(num_threads)))

    return sum(resultados)

# Testar a implementação com blocos de tamanhos muito variados
random.seed(42)
blocos_variados = [[j] * random.randint(10, 5000) for j in range(100)]
resultado = prefix_sums_balance(blocos_variados, num_threads=4)
print(f"Soma balanceada por prefix sums: {resultado}")


### Exercício 24: Busca Paralela em Vetor Ordenado
Implementar busca binária paralela, ativando o paralelismo apenas quando o tamanho do vetor é suficientemente grande para mitigar o overhead de criação de threads.

In [ ]:
def busca_paralela_ordenada(arr, alvo, num_threads=4):
    n = len(arr)
    limite = 100000  # Só paraleliza se for maior que 100.000 elementos

    def busca_binaria(inicio, fim):
        esq, dir = inicio, fim
        while esq <= dir:
            meio = (esq + dir) // 2
            if arr[meio] == alvo:
                return meio
            elif arr[meio] < alvo:
                esq = meio + 1
            else:
                dir = meio - 1
        return -1

    if n <= limite:
        # Execução sequencial direta
        print("Modo de busca sequencial ativo.")
        return busca_binaria(0, n - 1)

    # Execução paralela: divide o vetor ordenado em subvetores
    print("Modo de busca paralela ativo.")
    tamanho_bloco = (n + num_threads - 1) // num_threads

    def buscar_na_thread(id_thread):
        inicio = id_thread * tamanho_bloco
        fim = min(inicio + tamanho_bloco - 1, n - 1)
        # Otimização: busca apenas se o alvo estiver dentro da faixa desse bloco
        if arr[inicio] <= alvo <= arr[fim]:
            return busca_binaria(inicio, fim)
        return -1

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        resultados = list(executor.map(buscar_na_thread, range(num_threads)))

    for res in resultados:
        if res != -1:
            return res
    return -1

# Testar a busca
vetor_ordenado = list(range(200000))
pos_alvo = busca_paralela_ordenada(vetor_ordenado, 123456, num_threads=4)
print(f"Elemento encontrado na posição: {pos_alvo}")


### Exercício 25: Multiplicação Paralela de Matrizes
Implementar multiplicação de matrizes de forma paralela, mapeando a computação de linhas independentes da matriz resultante para diferentes threads.

In [ ]:
def multiplicacao_matrizes_paralela(A, B, num_threads=4):
    N = len(A)
    # Matriz resultante inicializada com zeros
    C = [[0] * N for _ in range(N)]

    # Função que calcula a multiplicação para um subconjunto de linhas
    def multiplicar_bloco_linhas(inicio_linha):
        linhas_por_thread = (N + num_threads - 1) // num_threads
        fim_linha = min(inicio_linha + linhas_por_thread, N)
        for i in range(inicio_linha, fim_linha):
            for j in range(N):
                soma = 0
                for k in range(N):
                    soma += A[i][k] * B[k][j]
                C[i][j] = soma

    # Distribuir as linhas iniciais para cada thread
    linhas_por_thread = (N + num_threads - 1) // num_threads
    inicios = [i * linhas_por_thread for i in range(num_threads)]

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        list(executor.map(multiplicar_bloco_linhas, inicios))

    return C

# Testar com duas matrizes 100 x 100
matriz_A = [[1] * 100 for _ in range(100)]
matriz_B = [[2] * 100 for _ in range(100)]
t_ini = time.time()
matriz_C = multiplicacao_matrizes_paralela(matriz_A, matriz_B, num_threads=4)
print(f"Multiplicação paralela (100x100) concluída | Tempo: {time.time() - t_ini:.4f} s | Valor C[0][0]: {matriz_C[0][0]}")
